# Control Engineering RAG — Exploration Notebook

This notebook is for understanding and improving the RAG pipeline.
We analyse chunk quality, embedding behaviour, and retrieval performance.

In [1]:
import sys
print(sys.executable)

C:\Users\yabdr\anaconda3\python.exe


In [2]:
C:\Users\yabdr\AppData\Local\Programs\Python\Python314\python.exe -m pip install pymupdf sentence-transformers chromadb sympy streamlit requests python-dotenv jupyter ipykernel pandas matplotlib

SyntaxError: unexpected character after line continuation character (1993550531.py, line 1)

In [1]:
import sys
sys.path.append('..')   # lets the notebook import from src/

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from src.pipeline.pdf_extractor import extract_all_pdfs
from src.pipeline.chunker import chunk_pages
from src.pipeline.indexer import query_collection

ModuleNotFoundError: No module named 'fitz'

In [ ]:
pages = extract_all_pdfs("../data/pdfs")
chunks = chunk_pages(pages)

print(f"Total pages : {len(pages)}")
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0])

In [ ]:
chunk_lengths = [len(c["text"]) for c in chunks]

plt.figure(figsize=(10, 4))
plt.hist(chunk_lengths, bins=30, color="steelblue", edgecolor="white")
plt.axvline(x=512, color="red", linestyle="--", label="Target chunk size (512)")
plt.title("Distribution of Chunk Lengths")
plt.xlabel("Characters")
plt.ylabel("Number of chunks")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean chunk length : {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")
print(f"Min chunk length  : {min(chunk_lengths)} chars")
print(f"Max chunk length  : {max(chunk_lengths)} chars")

In [ ]:
source_counts = Counter(c["source"] for c in chunks)
df = pd.DataFrame(source_counts.items(), columns=["PDF", "Chunks"])
df = df.sort_values("Chunks", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df["PDF"], df["Chunks"], color="steelblue")
plt.title("Chunks per PDF")
plt.xlabel("Number of chunks")
plt.tight_layout()
plt.show()

In [ ]:
test_questions = [
    "What is a transfer function?",
    "What is the Routh-Hurwitz criterion?",
    "What is state space representation?",
    "How does a PID controller work?",
]

for q in test_questions:
    results = query_collection(q, n_results=2)
    print(f"\nQ: {q}")
    for r in results:
        print(f"  [{r['distance']:.3f}] {r['source']} p.{r['page']}")